In [6]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import matplotlib.lines as mlines

# ==========================================
# تنظیمات استایل برای مقالات Q1
# ==========================================
plt.rcParams.update({
    'font.size': 20,
    'axes.titlesize': 26,
    'axes.labelsize': 22,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 22,
    'lines.linewidth': 3.5,
    'lines.markersize': 12,
    'font.family': 'sans-serif'
})

base_dir = "/home/ubuntu/Projects/thesis-code/results/baseline/final_linear_v4"

files_mapping = {
    "Clinical Text": f"{base_dir}/text_only/history.json",
    "Photograph-only": f"{base_dir}/image_only/history.json",
    "Radiograph-only": f"{base_dir}/xray_only/history.json",
    "Photograph + Radiograph": f"{base_dir}/image_xray/history.json",
    "Photograph + Text": f"{base_dir}/image_text/history.json",
    "Radiograph + Text": f"{base_dir}/text_xray/history.json",
    "Full Multimodal": f"{base_dir}/full_multimodal/history.json"
}

COLOR_TRAIN = '#1f77b4'
COLOR_VAL = '#ff7f0e'

def load_history(filepath):
    if not os.path.exists(filepath):
        print(f"Warning: File not found -> {filepath}")
        return None
    with open(filepath, 'r', encoding='utf-8') as f:
        return json.load(f)

def main():
    fig, axes = plt.subplots(nrows=7, ncols=2, figsize=(32, 55), dpi=300)
    
    handles_f1, labels_f1 = None, None
    handles_loss, labels_loss = None, None

    # پراکسی برای خط چین عمودی جهت نمایش در راهنما (Legend)
    vline_proxy = mlines.Line2D([], [], color='gray', linestyle='--', linewidth=2.5, label='Selected Checkpoint (Max Val F1)')

    for i, (scenario_name, filename) in enumerate(files_mapping.items()):
        ax_f1 = axes[i, 0]
        ax_loss = axes[i, 1]
        
        history = load_history(filename)
        
        if history is None:
            ax_f1.set_title(f"{scenario_name} (Macro-F1)\nData Not Found")
            ax_loss.set_title(f"{scenario_name} (Loss)\nData Not Found")
            ax_f1.axis('off')
            ax_loss.axis('off')
            continue

        epochs = [item['epoch'] for item in history]
        train_f1 = [item['train_macro_f1'] for item in history]
        val_f1 = [item['validation_macro_f1'] for item in history]
        train_loss = [item['train_loss'] for item in history]
        val_loss = [item['validation_loss'] for item in history]

        # استخراج مقادیر در بهترین ایپاک
        best_val_f1 = max(val_f1)
        idx_best = val_f1.index(best_val_f1)
        best_epoch = epochs[idx_best]
        
        best_train_f1 = train_f1[idx_best]
        best_val_loss = val_loss[idx_best]
        best_train_loss = train_loss[idx_best]

        props = dict(boxstyle='round,pad=0.6', facecolor='#f8f9fa', edgecolor='gray', alpha=1.0)

        # ==========================================
        # 1. رسم نمودار Macro-F1 (سمت چپ)
        # ==========================================
        line_t_f1, = ax_f1.plot(epochs, train_f1, label='Train Macro-F1', color=COLOR_TRAIN, alpha=0.9)
        line_v_f1, = ax_f1.plot(epochs, val_f1, label='Validation Macro-F1', color=COLOR_VAL, alpha=0.9)
        
        if handles_f1 is None:
            handles_f1 = [line_t_f1, line_v_f1, vline_proxy]
            labels_f1 = ['Train Macro-F1', 'Validation Macro-F1', 'Selected Checkpoint (Max Val F1)']

        # خط عمودی
        ax_f1.axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.7, linewidth=2.5)
        
        # نقطه‌گذاری روی هر دو خط در ایپاک بهینه
        ax_f1.scatter([best_epoch], [best_val_f1], color=COLOR_VAL, s=200, zorder=5, edgecolor='black')
        ax_f1.scatter([best_epoch], [best_train_f1], color=COLOR_TRAIN, s=200, zorder=5, edgecolor='black')
        
        text_str_f1 = f'Best Val F1 = {best_val_f1:.3f} (Epoch {best_epoch})'
        ax_f1.text(0.5, -0.22, text_str_f1, transform=ax_f1.transAxes, fontsize=24, fontweight='bold',
                   verticalalignment='top', horizontalalignment='center', bbox=props, clip_on=False)

        ax_f1.set_title(f"{scenario_name} (Macro-F1)", fontweight='bold', pad=15)
        ax_f1.set_xlabel('Epoch')
        ax_f1.set_ylabel('Macro-F1')
        ax_f1.set_xlim(0, 41)
        ax_f1.set_ylim(0.0, 1.05)
        ax_f1.grid(True, linestyle='--', alpha=0.4)

        # ==========================================
        # 2. رسم نمودار Loss (سمت راست)
        # ==========================================
        line_t_loss, = ax_loss.plot(epochs, train_loss, label='Train Loss', color=COLOR_TRAIN, alpha=0.9)
        line_v_loss, = ax_loss.plot(epochs, val_loss, label='Validation Loss', color=COLOR_VAL, alpha=0.9)
        
        if handles_loss is None:
            handles_loss = [line_t_loss, line_v_loss, vline_proxy]
            labels_loss = ['Train Loss', 'Validation Loss', 'Selected Checkpoint (Max Val F1)']

        # خط عمودی
        ax_loss.axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.7, linewidth=2.5)
        
        # نقطه‌گذاری روی هر دو خط در ایپاک بهینه
        ax_loss.scatter([best_epoch], [best_val_loss], color=COLOR_VAL, s=200, zorder=5, edgecolor='black')
        ax_loss.scatter([best_epoch], [best_train_loss], color=COLOR_TRAIN, s=200, zorder=5, edgecolor='black')

        text_str_loss = f'Val Loss at Epoch {best_epoch} = {best_val_loss:.4f}'
        ax_loss.text(0.5, -0.22, text_str_loss, transform=ax_loss.transAxes, fontsize=24, fontweight='bold',
                     verticalalignment='top', horizontalalignment='center', bbox=props, clip_on=False)

        ax_loss.set_title(f"{scenario_name} (Loss)", fontweight='bold', pad=15)
        ax_loss.set_xlabel('Epoch')
        ax_loss.set_ylabel('Loss')
        ax_loss.set_xlim(0, 41)
        ax_loss.set_ylim(bottom=0.0) 
        ax_loss.grid(True, linestyle='--', alpha=0.4)

    # ==========================================
    # تنظیمات کلی گرید و راهنمای رنگ سراسری
    # ==========================================
    # عرض تصویر را به 32 اینچ افزایش دادم تا ncol=3 (سه المان در Legend) جا بشوند
    fig.legend(handles_f1, labels_f1, loc='upper center', bbox_to_anchor=(0.28, 0.995), ncol=3, fontsize=22, framealpha=1.0)
    fig.legend(handles_loss, labels_loss, loc='upper center', bbox_to_anchor=(0.78, 0.995), ncol=3, fontsize=22, framealpha=1.0)

    plt.subplots_adjust(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.65, wspace=0.15)

    output_filename = 'learning_curves_and_losses_4k_final.jpg'
    plt.savefig(output_filename, format='jpg', bbox_inches='tight', dpi=300)
    print(f"Plot saved successfully as {output_filename}")

if __name__ == "__main__":
    main()

Plot saved successfully as learning_curves_and_losses_4k_final.jpg
